# Driver Status Classifier — Live Demo (Colab)
Runs the trained model using your browser's webcam. No training needed.

**Before running:** put `best_model.pth` in Google Drive at:
```
MyDrive/driver_status_classifier/models/best_model.pth
```

## 1 — Setup

In [ ]:
import os
REPO_URL = "https://github.com/DSKing123555/driver_status_classifier.git"
if not os.path.exists("/content/driver_status_classifier"):
    !git clone {REPO_URL} /content/driver_status_classifier

%cd /content/driver_status_classifier
!pip install -q mediapipe opencv-python-headless torch torchvision

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2 — Load model from Drive

In [ ]:
import shutil, pathlib, sys
sys.path.append("/content/driver_status_classifier/src")

# Copy model from Drive into the project folder
src = pathlib.Path("/content/drive/MyDrive/driver_status_classifier/models/best_model.pth")
dst = pathlib.Path("/content/driver_status_classifier/models/best_model.pth")
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(src, dst)

import torch
import torch.nn.functional as F
from config import MODELS_DIR, SEQUENCE_LENGTH, CLASSES
from model import DrowsinessLSTM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

raw = torch.load(dst, map_location=device)
class_names = raw.get("class_names", list(CLASSES))
model = DrowsinessLSTM(
    input_size=raw.get("num_features", 7),
    hidden_size=raw.get("hidden_size", 128),
    num_layers=raw.get("num_layers", 2),
    num_classes=len(class_names)
).to(device)
model.load_state_dict(raw["state_dict"])
model.eval()
print(f"Model loaded. Classes: {class_names}")

## 3 — Live webcam demo
The cell below opens your browser camera and runs the model in real time.
**Press the Stop button (■) in Colab to quit.**

In [ ]:
import base64, cv2, numpy as np, mediapipe as mp
from collections import deque
from IPython.display import display, Javascript, Image, clear_output
from google.colab.output import eval_js

sys.path.append("/content/driver_status_classifier/src")
from features import compute_ear_mar, compute_head_pose, make_face_detector
from config import RIGHT_EYE_INDICES, LEFT_EYE_INDICES, MOUTH_INDICES

# ── helpers ──────────────────────────────────────────────────────────────
def _b64_to_frame(b64):
    binary = base64.b64decode(b64.split(",")[1])
    return cv2.imdecode(np.frombuffer(binary, dtype=np.uint8), cv2.IMREAD_COLOR)

def _frame_to_jpeg_b64(frame):
    _, buf = cv2.imencode(".jpg", frame)
    return base64.b64encode(buf).decode()

# JavaScript that starts the webcam once and keeps it running
START_JS = """
(async () => {
  if (window._dms_stream) return 'already running';
  const v = document.createElement('video');
  v.style.display = 'none';
  document.body.appendChild(v);
  const stream = await navigator.mediaDevices.getUserMedia(
      {video: {width: 640, height: 480}});
  v.srcObject = stream;
  await v.play();
  window._dms_video  = v;
  window._dms_stream = stream;
  return 'started';
})()
"""

CAPTURE_JS = """
(async () => {
  const v = window._dms_video;
  const c = document.createElement('canvas');
  c.width  = v.videoWidth  || 640;
  c.height = v.videoHeight || 480;
  c.getContext('2d').drawImage(v, 0, 0);
  return c.toDataURL('image/jpeg', 0.8);
})()
"""

STOP_JS = """
(async () => {
  if (window._dms_stream) {
    window._dms_stream.getTracks().forEach(t => t.stop());
    window._dms_video.remove();
    delete window._dms_stream;
    delete window._dms_video;
  }
  return 'stopped';
})()
"""

# ── inference state ───────────────────────────────────────────────────────
DROWSY_MIN_PROB = 0.60
SWITCH_FRAMES   = 30

feature_buffer = deque(maxlen=SEQUENCE_LENGTH)
pred_buffer    = deque(maxlen=45)
ear_history    = deque(maxlen=5)
last_valid     = [0.0] * 5

stable_label    = None
stable_conf     = 0.0
candidate_label = None
candidate_frames = 0

detector = make_face_detector()

# ── start camera ─────────────────────────────────────────────────────────
eval_js(START_JS)
print("Camera started. Press Stop (■) to quit.")

try:
    while True:
        # Capture one frame from the browser webcam
        b64 = eval_js(CAPTURE_JS)
        frame = _b64_to_frame(b64)
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = detector.detect(mp_image)

        # Feature extraction
        if result.face_landmarks:
            lms = result.face_landmarks[0]
            ear, mar = compute_ear_mar(lms, w, h)
            yaw, pitch, roll = compute_head_pose(lms, w, h)
            ear_history.append(ear)
            smoothed_ear = float(np.mean(ear_history))
            last_valid = [smoothed_ear, mar, yaw, pitch, roll]
            cv2.putText(frame, f"EAR:{smoothed_ear:.3f}  MAR:{mar:.3f}",
                        (10, h - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
        else:
            cv2.putText(frame, "No face", (10, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

        feature_buffer.append(list(last_valid))

        label_text = f"Buffering {len(feature_buffer)}/{SEQUENCE_LENGTH}"
        conf_text  = ""

        if len(feature_buffer) == SEQUENCE_LENGTH:
            buf = np.array(list(feature_buffer), dtype=np.float32)  # (60,5)
            delta = np.zeros((SEQUENCE_LENGTH, 2), dtype=np.float32)
            delta[1:] = buf[1:, :2] - buf[:-1, :2]
            buf7 = np.concatenate([buf, delta], axis=1)              # (60,7)

            x = torch.tensor([buf7], dtype=torch.float32, device=device)
            with torch.no_grad():
                probs = F.softmax(model(x), dim=1)[0].cpu().numpy()

            pred_buffer.append(probs)
            avg = np.mean(list(pred_buffer), axis=0)

            raw_idx = int(avg.argmax())
            if class_names[raw_idx] == "Drowsy" and avg[raw_idx] < DROWSY_MIN_PROB:
                top_idx = class_names.index("Alert")
            else:
                top_idx = raw_idx
            top_label = class_names[top_idx]
            top_conf  = float(avg[top_idx])

            # Hysteresis
            if stable_label is None:
                stable_label = top_label; stable_conf = top_conf
            elif top_label != stable_label:
                if top_label == candidate_label:
                    candidate_frames += 1
                else:
                    candidate_label = top_label; candidate_frames = 1
                if candidate_frames >= SWITCH_FRAMES:
                    stable_label = candidate_label; stable_conf = top_conf
                    candidate_label = None; candidate_frames = 0
            else:
                stable_conf = top_conf
                candidate_label = None; candidate_frames = 0

            label_text = stable_label
            conf_text  = f"{stable_conf*100:.0f}%"

            # Bar chart (right side)
            bar_max = 200; x0 = w - bar_max - 10; y0 = 10; bh = 18
            for i, name in enumerate(class_names):
                y = y0 + i*(bh+4)
                cv2.rectangle(frame, (x0,y), (x0+bar_max, y+bh), (50,50,50), -1)
                cv2.rectangle(frame, (x0,y),
                              (x0+int(bar_max*avg[i]), y+bh),
                              (0,200,0) if i==top_idx else (180,180,180), -1)
                cv2.putText(frame, f"{name} {avg[i]*100:.0f}%",
                            (x0+4, y+bh-4), cv2.FONT_HERSHEY_SIMPLEX, 0.45,
                            (255,255,255), 1)

        # Big label overlay
        color = (0,0,200) if label_text == "Drowsy" else (0,200,0)
        cv2.rectangle(frame, (5,5), (350,65), (0,0,0), -1)
        cv2.putText(frame, f"{label_text}  {conf_text}",
                    (15, 52), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

        # Show frame in notebook
        clear_output(wait=True)
        display(Image(data=base64.b64decode(_frame_to_jpeg_b64(frame))))

except KeyboardInterrupt:
    pass
finally:
    detector.close()
    eval_js(STOP_JS)
    print("Camera stopped.")